# 5.6 Belief Propagation

Inference becomes a conversation: each message is a summarized view of one side of the graph.

Belief propagation uses factor graphs and exact marginalization. It powers HMM filtering, CRFs, and becomes exact on junction trees.

Save a copy to Drive to edit.

In [ ]:

import itertools
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(5606)


def normalize(values):
    values = np.asarray(values, dtype=float)
    total = values.sum()
    return values / total


def sum_product_chain(pi, transitions, evidence):
    n = len(evidence)
    forward = [normalize(pi * evidence[0])]
    for t in range(1, n):
        message = forward[-1] @ transitions[t - 1]
        forward.append(normalize(message * evidence[t]))
    backward = [np.ones_like(pi) for _ in range(n)]
    for t in range(n - 2, -1, -1):
        message = transitions[t] @ (evidence[t + 1] * backward[t + 1])
        backward[t] = normalize(message)
    beliefs = []
    for t in range(n):
        beliefs.append(normalize(forward[t] * backward[t]))
    return beliefs, forward, backward


def exact_pairwise_marginals(unaries, edges, pairwise):
    n = len(unaries)
    assignments = list(itertools.product([0, 1], repeat=n))
    scores = []
    for assignment in assignments:
        score = 1.0
        for i, value in enumerate(assignment):
            score = score * unaries[i][value]
        for i, j in edges:
            score = score * pairwise[assignment[i], assignment[j]]
        scores.append(score)
    probs = normalize(np.asarray(scores))
    marginals = np.zeros((n, 2), dtype=float)
    for assignment, probability in zip(assignments, probs):
        for i, value in enumerate(assignment):
            marginals[i, value] = marginals[i, value] + probability
    return marginals


def loopy_bp(unaries, edges, pairwise, iterations=20, damping=0.0):
    n = len(unaries)
    directed = []
    neighbors = {i: [] for i in range(n)}
    for i, j in edges:
        directed.append((i, j))
        directed.append((j, i))
        neighbors[i].append(j)
        neighbors[j].append(i)
    messages = {edge: np.ones(2) / 2.0 for edge in directed}
    history = []
    for step in range(iterations):
        new_messages = {}
        for i, j in directed:
            incoming = unaries[i].copy()
            for k in neighbors[i]:
                if k != j:
                    incoming = incoming * messages[(k, i)]
            raw = incoming @ pairwise
            raw = normalize(raw)
            new_messages[(i, j)] = damping * messages[(i, j)] + (1.0 - damping) * raw
        messages = new_messages
        beliefs = []
        for i in range(n):
            belief = unaries[i].copy()
            for k in neighbors[i]:
                belief = belief * messages[(k, i)]
            beliefs.append(normalize(belief))
        history.append(np.asarray(beliefs))
    return history


def mean_marginal_error(estimate, truth):
    return float(np.mean(np.abs(np.asarray(estimate) - np.asarray(truth))))


## The concept, built once (D1)

Sum-product sends a factor-to-variable message by summing over the other variables in that factor.

$$m_{a\to i}(x_i)=\sum_{x_a\setminus x_i} f_a(x_a)\prod_{j\in a\setminus i} m_{j\to a}(x_j)$$

The chain implementation is exact on trees; the loopy implementation later uses the same exclusion rule with iterative updates.

In [ ]:

def d1_messages():
    transition = np.array([[0.9, 0.1], [0.2, 0.8]])
    right_emission = np.array([0.3, 0.7])
    left_prior = np.array([0.6, 0.4])
    right_message = transition @ right_emission
    left_contribution = left_prior @ transition
    scores = left_contribution * right_message
    belief = normalize(scores)
    return right_message, left_contribution, scores, belief


The lesson messages are right-to-$Y$ $[0.340,0.620]$, left contribution $[0.620,0.380]$, belief scores $[0.2108,0.2356]$, and normalized $p(Y=0)=0.472222$.

In [ ]:

right_message, left_contribution, scores, belief = d1_messages()
assert np.allclose(right_message, np.array([0.34, 0.62]))
assert np.allclose(left_contribution, np.array([0.62, 0.38]))
assert np.allclose(scores, np.array([0.2108, 0.2356]))
assert abs(belief[0] - 0.4722222222222222) < 1e-12
print("right message", right_message)
print("left contribution", left_contribution)
print("belief", belief)


## The dataset ladder

The ladder moves from an exact three-node chain to a four-state tree, one loopy graph, a small real sequence-style contingency table, and a higher-dimensional loopy grid with damped updates.

In [ ]:

rungs = []

pi = np.array([0.6, 0.4])
transitions = [np.array([[0.9, 0.1], [0.2, 0.8]]), np.array([[0.9, 0.1], [0.2, 0.8]])]
evidence = [np.ones(2), np.ones(2), np.array([0.3, 0.7])]
beliefs, forward, backward = sum_product_chain(pi, transitions, evidence)
rungs.append({"name": "D1 chain", "truth": np.asarray(beliefs), "estimate": np.asarray(beliefs), "iterations": [0.0]})

pi = np.array([0.25, 0.25, 0.25, 0.25])
T = np.array([[0.7, 0.1, 0.1, 0.1], [0.1, 0.7, 0.1, 0.1], [0.1, 0.1, 0.7, 0.1], [0.1, 0.1, 0.1, 0.7]])
evidence = [np.ones(4), np.array([1.0, 0.7, 0.4, 0.2]), np.array([0.3, 0.6, 1.0, 0.5])]
beliefs, forward, backward = sum_product_chain(pi, [T, T], evidence)
rungs.append({"name": "D2 four-state tree", "truth": np.asarray(beliefs), "estimate": np.asarray(beliefs), "iterations": [0.0]})

unaries = [np.array([0.55, 0.45]) for _ in range(4)]
edges = [(0, 1), (1, 2), (2, 3), (3, 0)]
pairwise = np.array([[1.8, 0.4], [0.4, 1.8]])
truth = exact_pairwise_marginals(unaries, edges, pairwise)
history = loopy_bp(unaries, edges, pairwise, iterations=12, damping=0.2)
rungs.append({"name": "D3 single loop", "truth": truth, "estimate": history[-1], "iterations": [mean_marginal_error(item, truth) for item in history]})

unaries = [np.array([0.8, 0.2]), np.array([0.35, 0.65]), np.array([0.45, 0.55]), np.array([0.25, 0.75]), np.array([0.7, 0.3])]
edges = [(0, 1), (1, 2), (2, 3), (3, 4)]
pairwise = np.array([[1.6, 0.5], [0.5, 1.6]])
truth = exact_pairwise_marginals(unaries, edges, pairwise)
history = loopy_bp(unaries, edges, pairwise, iterations=5, damping=0.0)
rungs.append({"name": "D4 sequence table", "truth": truth, "estimate": history[-1], "iterations": [mean_marginal_error(item, truth) for item in history]})

unaries = []
for value in rng.integers(0, 2, size=9):
    unaries.append(np.array([2.0, 0.5]) if value == 0 else np.array([0.5, 2.0]))
edges = []
for r in range(3):
    for c in range(3):
        idx = r * 3 + c
        if r < 2:
            edges.append((idx, idx + 3))
        if c < 2:
            edges.append((idx, idx + 1))
pairwise = np.array([[2.2, 0.3], [0.3, 2.2]])
truth = exact_pairwise_marginals(unaries, edges, pairwise)
history = loopy_bp(unaries, edges, pairwise, iterations=30, damping=0.5)
rungs.append({"name": "D5 loopy grid", "truth": truth, "estimate": history[-1], "iterations": [mean_marginal_error(item, truth) for item in history]})

for rung in rungs:
    print(rung["name"], "nodes", rung["truth"].shape[0], "sample", np.round(rung["truth"][:3], 3))


In [ ]:

errors = []
for rung in rungs:
    err = mean_marginal_error(rung["estimate"], rung["truth"])
    errors.append(err)
    print(f"{rung['name']}: marginal_error={err:.6f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for i, rung in enumerate(rungs):
    x = np.arange(rung["truth"].shape[0])
    axes[0, i].bar(x - 0.2, rung["truth"][:, 1], width=0.4, label="exact")
    axes[0, i].bar(x + 0.2, rung["estimate"][:, 1], width=0.4, label="BP")
    axes[0, i].set_ylim(0, 1)
    axes[0, i].set_title(rung["name"])
    axes[1, i].axis("off")
axes[0, 0].legend()
axes[1, 2].plot(range(1, 6), errors, marker="o", label="final")
axes[1, 2].plot(range(1, len(rungs[-1]["iterations"]) + 1), rungs[-1]["iterations"], marker=".", label="D5 curve")
axes[1, 2].set_xlabel("rung or iteration")
axes[1, 2].set_ylabel("marginal error")
axes[1, 2].set_title("error vs iteration")
axes[1, 2].legend()
fig.tight_layout()


## Pitfall on D5: trusting loopy beliefs blindly

On a tree, messages are exact. On a grid, reusing neighbor evidence without exclusion or without damping can oscillate or double-count. Damping and correct exclusion improve stability, but exact enumeration remains the reference for this small D5.

In [ ]:

d5_unaries = unaries
undamped_history = loopy_bp(d5_unaries, edges, pairwise, iterations=8, damping=0.0)
damped_history = loopy_bp(d5_unaries, edges, pairwise, iterations=8, damping=0.5)
undamped_error = mean_marginal_error(undamped_history[-1], truth)
damped_error = mean_marginal_error(damped_history[-1], truth)
print("undamped error", undamped_error)
print("damped error", damped_error)
print("exact reference first node", truth[0])
assert np.isfinite(undamped_error)
assert np.isfinite(damped_error)


## Evaluate it + Practice

- Metric: mean marginal error versus exact enumeration or exact chain/tree sum-product.
- No-skill baseline: use the prior, unary factors, or a single local conditional without global inference.
- Cheap sanity check: messages and beliefs normalize to one and tree cases match exact enumeration.
- Ablation: turn off neighbor-exclusion or damping on D5 and compare errors over iterations.
- Failure signals: oscillating messages, impossible probabilities, or confusing max-product MAP with marginals.

Practice prompts:
1. Add evidence to the middle node of D1.

2. Compare sum-product and max-product decisions on D4.

3. Tune damping on D5 and plot the error curve.